<a href="https://colab.research.google.com/github/KhangLy2606/trade-bot-colab-notebooks/blob/master/regime_classifier_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multi-Timeframe Return Training — Trade Bot

Trains LightGBM models to predict next-bar return **direction** (classification) and **magnitude** (regression) across 5m, 15m, and 1h candles using Massive.com S3 minute-bar data.

**Pipeline:**
```
Massive S3 flat-files (1m bars, multiple symbols)
  → Resample to 5m / 15m / 1h candles
  → Timeframe-aware rolling features (wall-clock windows)
  → Next-bar return targets (direction + magnitude)
  → Walk-forward CV (timeframe-scaled gap + test window)
  → Persistence/random-walk baselines
  → Compare 5m vs 15m vs 1h
  → Save best to Google Drive → register in MLflow
```

**Key changes from regime classifier:**
- **Target**: next-bar return (not regime labels)
- **Features**: window sizes scale by wall-clock time, not bar count
- **Validation**: gap and test window in hours/days, not fixed bars
- **Baselines**: every model is compared to persistence and random-walk
- **Multi-symbol**: pools 12 liquid symbols for 100k+ rows per timeframe

**Runtime:** CPU is fine. LightGBM does not use GPU for this task.

---
### Credentials
Resolved in order: Colab Secrets → env vars → `.env` file → fallback

In [1]:
# ── Verify we are running on Colab ────────────────────────────────────────────
import sys

IN_COLAB = 'google.colab' in sys.modules
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Running on Colab: {IN_COLAB}')
print(f'Python: {sys.version}')

Running on Colab: True
Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]


In [2]:
# ── Install dependencies (run this FIRST before any imports) ──────────────────
# Must run before importing scipy/sklearn so Python loads the upgraded versions
# fresh from disk. No kernel restart needed when this is the first cell executed.
#
# Why these pins:
#   scipy>=1.15, scikit-learn>=1.6 — first releases supporting numpy 2.2
#                                    (Colab ships numpy 2.2.6 system-wide)
#   fsspec==2025.3.0, s3fs==2025.3.0 — must match exactly or s3fs fails
#   nolds, pandas-ta excluded        — no binary wheels for Python 3.12;
#                                      pure-numpy fallbacks used instead
%pip install -q --prefer-binary \
    "scipy>=1.15" \
    "scikit-learn>=1.6" \
    "fsspec==2025.3.0" \
    "s3fs==2025.3.0" \
    "lightgbm>=4.0" \
    "optuna>=3.0" \
    "mlflow>=2.0" \
    "python-dotenv>=1.0"

print("Install complete — proceed to next cell.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 89.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.3/87.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.1/197.1 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.3/14.3 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.5/

In [ ]:
# ── Restart runtime to activate newly installed packages ──────────────────────
# IMPORTANT — read before running:
#
#   1. Run this cell ONCE, immediately after the install cell above finishes.
#   2. The kernel will restart. VS Code shows "The Kernel crashed" — this is
#      EXPECTED and harmless. It is not a real crash.
#   3. After the restart, use Runtime → Run all (or Ctrl+F9) to execute ALL
#      cells from the top. Do NOT manually run cells one by one — variables
#      like IN_COLAB, SAVE_DIR, and S3_CONFIG will be undefined and you will
#      get NameError on the first cell that references them.
#
# Why IPython.do_shutdown instead of os.kill(os.getpid(), 9)?
#   os.kill sends SIGKILL which terminates pip mid-install if it is still
#   running. do_shutdown cooperatively restarts only the kernel process.

import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)

{'status': 'ok', 'restart': True}

: 

In [ ]:
# ── Load credentials ───────────────────────────────────────────────────────────
# Strategy (tried in order):
#   1. Colab Secrets panel (browser Colab UI, most secure)
#   2. Environment variables
#   3. .env file search (Drive + workspace paths)
#   4. Hardcoded fallback (last resort)
#
# Also sets up SAVE_DIR for Google Drive storage

import os
from pathlib import Path

MASSIVE_ACCESS_KEY = None
MASSIVE_SECRET_KEY = None

# Secrets can be named either MASSIVE_ACCESS_KEY / MASSIVE_SECRET_KEY
# or MASSIVE_S3_ACCESS_KEY / MASSIVE_S3_SECRET_KEY depending on setup.
SECRET_ACCESS_NAMES = ['MASSIVE_ACCESS_KEY', 'MASSIVE_S3_ACCESS_KEY']
SECRET_SECRET_NAMES = ['MASSIVE_SECRET_KEY', 'MASSIVE_S3_SECRET_KEY']

print(f'IN_COLAB={IN_COLAB}')

# ── Layer 1: Colab Secrets panel ───────────────────────────────────────────────
if IN_COLAB:
    try:
        from google.colab import userdata

        ui_only_secret_error = False

        # Try both secret naming conventions
        for access_name in SECRET_ACCESS_NAMES:
            try:
                val = userdata.get(access_name)
                if val:
                    MASSIVE_ACCESS_KEY = val
                    print(f'✅ Access key from Colab Secrets: {access_name}')
                    break
            except Exception as e:
                if 'only be fetched when running from the Colab UI' in str(e):
                    ui_only_secret_error = True
                continue

        for secret_name in SECRET_SECRET_NAMES:
            try:
                val = userdata.get(secret_name)
                if val:
                    MASSIVE_SECRET_KEY = val
                    print(f'✅ Secret key from Colab Secrets: {secret_name}')
                    break
            except Exception as e:
                if 'only be fetched when running from the Colab UI' in str(e):
                    ui_only_secret_error = True
                continue

        if ui_only_secret_error:
            print('⚠️  Colab Secrets are UI-only in this session.')
            print('   You are likely connected via VS Code/extension runtime, not browser Colab UI.')
            print('   Use .env or environment variables for this session.')
        elif not MASSIVE_ACCESS_KEY or not MASSIVE_SECRET_KEY:
            print('⚠️  Colab Secrets available but Massive keys not resolved for this notebook session')
            print('   Verify both secrets exist and Notebook access is enabled')
    except Exception as e:
        print(f'⚠️  Colab userdata unavailable in this runtime: {type(e).__name__}')

# ── Layer 2: Environment variables ─────────────────────────────────────────────
if not MASSIVE_ACCESS_KEY or not MASSIVE_SECRET_KEY:
    # Support both env naming conventions
    MASSIVE_ACCESS_KEY = MASSIVE_ACCESS_KEY or os.getenv('MASSIVE_S3_ACCESS_KEY') or os.getenv('MASSIVE_ACCESS_KEY')
    MASSIVE_SECRET_KEY = MASSIVE_SECRET_KEY or os.getenv('MASSIVE_S3_SECRET_KEY') or os.getenv('MASSIVE_SECRET_KEY')
    if MASSIVE_ACCESS_KEY and MASSIVE_SECRET_KEY:
        print('✅ Credentials from environment variables')

# ── Layer 3: .env search (runtime-agnostic) ───────────────────────────────────
if not MASSIVE_ACCESS_KEY or not MASSIVE_SECRET_KEY:
    try:
        from dotenv import load_dotenv

        search_paths = [
            Path.cwd(),
            Path.cwd().parent,
            Path.cwd().parent.parent,
            Path('/home/opc/projects/trade-bot/market-dashboard/backend/notebooks'),
        ]

        if IN_COLAB:
            drive_paths = [
                Path('/content/drive/MyDrive/trade-bot/market-dashboard/backend/notebooks'),
                Path('/content/drive/MyDrive/trade-bot/backend/notebooks'),
                Path('/content/drive/MyDrive/trade-bot/market-dashboard'),
                Path('/content/drive/MyDrive/trade-bot'),
                Path('/content/drive/MyDrive'),
                Path('/content'),
            ]
            search_paths = drive_paths + search_paths

        # Deduplicate while preserving order
        deduped_paths = []
        seen = set()
        for p in search_paths:
            key = str(p)
            if key not in seen:
                deduped_paths.append(p)
                seen.add(key)

        env_found = False
        loaded_from = None
        for search_dir in deduped_paths:
            env_file = search_dir / '.env'
            if env_file.exists():
                try:
                    load_dotenv(env_file, override=False)
                    MASSIVE_ACCESS_KEY = MASSIVE_ACCESS_KEY or os.getenv('MASSIVE_S3_ACCESS_KEY') or os.getenv('MASSIVE_ACCESS_KEY')
                    MASSIVE_SECRET_KEY = MASSIVE_SECRET_KEY or os.getenv('MASSIVE_S3_SECRET_KEY') or os.getenv('MASSIVE_SECRET_KEY')
                    if MASSIVE_ACCESS_KEY and MASSIVE_SECRET_KEY:
                        env_found = True
                        loaded_from = env_file
                        break
                except Exception:
                    continue

        if env_found:
            print(f'✅ Credentials from .env file: {loaded_from}')
        elif not MASSIVE_ACCESS_KEY or not MASSIVE_SECRET_KEY:
            print('⚠️  .env file not found in standard locations')
            print('   Checked:')
            for p in deduped_paths:
                status = '✓' if (p / '.env').exists() else '✗'
                print(f'      {status} {p}')
    except ImportError:
        print('⚠️  python-dotenv not available, skipping .env layer')
    except Exception as e:
        print(f'⚠️  Error loading .env: {type(e).__name__}: {str(e)[:120]}')

# ── Layer 4: Hardcoded fallback ────────────────────────────────────────────────
if not MASSIVE_ACCESS_KEY or not MASSIVE_SECRET_KEY:
    MASSIVE_ACCESS_KEY = MASSIVE_ACCESS_KEY or 'YOUR_MASSIVE_ACCESS_KEY_HERE'
    MASSIVE_SECRET_KEY = MASSIVE_SECRET_KEY or 'YOUR_MASSIVE_SECRET_KEY_HERE'
    print('⚠️  Using hardcoded fallback credentials')
    print()
    print('HOW TO FIX — Choose ONE option:')
    print()
    if IN_COLAB:
        print('Option 1: Colab Secrets panel (recommended)')
        print('  - Open 🔑 Secrets panel in BROWSER Colab UI')
        print('  - Add either pair:')
        print('      MASSIVE_ACCESS_KEY + MASSIVE_SECRET_KEY')
        print('      OR MASSIVE_S3_ACCESS_KEY + MASSIVE_S3_SECRET_KEY')
        print('  - Enable Notebook access for both')
        print('  - Re-run this cell from browser Colab UI')
        print()
        print('Option 2: Put .env on Drive at one of these paths:')
        print('  - /content/drive/MyDrive/trade-bot/market-dashboard/backend/notebooks/.env')
        print('  - /content/drive/MyDrive/trade-bot/.env')
        print('  - /content/drive/MyDrive/.env')
        print()
    else:
        print('Option 1: Create .env in notebooks folder from .env.example')
        print('Option 2: Set env vars MASSIVE_S3_ACCESS_KEY and MASSIVE_S3_SECRET_KEY')

S3_CONFIG = {
    'endpoint': os.getenv('MASSIVE_S3_ENDPOINT', 'https://files.massive.com'),
    'bucket': os.getenv('MASSIVE_S3_BUCKET', 'flatfiles'),
    'access_key': MASSIVE_ACCESS_KEY,
    'secret_key': MASSIVE_SECRET_KEY,
}

# ── Setup persistent directories ───────────────────────────────────────────────
if IN_COLAB:
    try:
        from google.colab import drive
        if not os.path.exists('/content/drive'):
            drive.mount('/content/drive')
        SAVE_DIR = '/content/drive/MyDrive/trade-bot-models'
    except Exception:
        SAVE_DIR = './trained-models'
else:
    SAVE_DIR = './trained-models'

CACHE_DIR = os.path.join(SAVE_DIR, 'cache')
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

print(f'Model storage: {SAVE_DIR}')
print(f'Cache storage: {CACHE_DIR}')

# Validation check
is_placeholder = (
    MASSIVE_ACCESS_KEY.startswith('YOUR_')
    or MASSIVE_SECRET_KEY.startswith('YOUR_')
)
print()
if is_placeholder:
    print('❌ CRITICAL: Placeholder credentials — S3 data loading WILL FAIL')
    print('             Please fix credentials using one of the options above')
else:
    print(f'✅ Access key loaded: {MASSIVE_ACCESS_KEY[:8]}...')
    print('✅ S3 config ready → You can run data loading cells')

IN_COLAB=True
✅ Access key from Colab Secrets: MASSIVE_ACCESS_KEY
✅ Secret key from Colab Secrets: MASSIVE_SECRET_KEY
Mounted at /content/drive
Model storage: /content/drive/MyDrive/trade-bot-models

✅ Access key loaded: bd65025c...
✅ S3 config ready → You can run data loading cells


In [4]:
# ── Credential Diagnostic — Run if credentials failed ────────────────────────
# This cell helps debug why credentials didn't load.

if is_placeholder:
    print('CREDENTIAL TROUBLESHOOTING GUIDE')
    print('=' * 70)
    print()
    print(f'Running on Colab: {IN_COLAB}')
    print(f'Current working directory: {Path.cwd()}')
    print()

    print('AVAILABLE CREDENTIAL SOURCES:')
    print('-' * 70)

    # Deep-check Colab Secrets access
    if IN_COLAB:
        try:
            from google.colab import userdata
            print('✅ Colab userdata API is available')

            secret_names = [
                'MASSIVE_ACCESS_KEY',
                'MASSIVE_SECRET_KEY',
                'MASSIVE_S3_ACCESS_KEY',
                'MASSIVE_S3_SECRET_KEY',
            ]
            print('Checking secret name resolution:')
            for name in secret_names:
                try:
                    val = userdata.get(name)
                    if val:
                        print(f'  ✅ {name}: FOUND (length={len(val)})')
                    else:
                        print(f'  ⚠️  {name}: empty/None')
                except Exception as e:
                    print(f'  ❌ {name}: {type(e).__name__}: {str(e)[:120]}')

            print()
            print('If you see PermissionError or no value above:')
            print('  1. Re-open Secrets panel (🔑)')
            print('  2. Ensure BOTH keys exist')
            print('  3. Ensure Notebook access toggle is ON for both')
            print('  4. Close and re-open notebook, then re-run cell')
        except Exception as e:
            print(f'✗ Colab Secrets API unavailable: {type(e).__name__}: {e}')
    else:
        print('✗ Not running on Colab (Secrets panel unavailable)')

    # Check env vars
    print()
    env_access = os.getenv('MASSIVE_S3_ACCESS_KEY') or os.getenv('MASSIVE_ACCESS_KEY')
    env_secret = os.getenv('MASSIVE_S3_SECRET_KEY') or os.getenv('MASSIVE_SECRET_KEY')
    if env_access and env_secret:
        print(f'✅ Environment variables set (access prefix: {env_access[:8]}...)')
    else:
        print('✗ Environment variables not set for Massive credentials')

    # Check .env files in various locations
    print()
    print('CHECKING FOR .env FILES:')
    print('-' * 70)

    search_paths = [
        Path('/content/drive/MyDrive/trade-bot/market-dashboard/backend/notebooks/.env'),
        Path('/content/drive/MyDrive/trade-bot/backend/notebooks/.env'),
        Path('/content/drive/MyDrive/trade-bot/.env'),
        Path('/content/drive/MyDrive/.env'),
        Path.cwd() / '.env',
        Path.cwd().parent / '.env',
        Path('/home/opc/projects/trade-bot/market-dashboard/backend/notebooks/.env'),
    ]

    for path in search_paths:
        if path.exists():
            print(f'✅ FOUND: {path}')
            try:
                with open(path) as f:
                    content = f.read()
                    has_access = ('MASSIVE_S3_ACCESS_KEY' in content) or ('MASSIVE_ACCESS_KEY' in content)
                    has_secret = ('MASSIVE_S3_SECRET_KEY' in content) or ('MASSIVE_SECRET_KEY' in content)
                    print(f'   └─ access key present: {has_access}, secret key present: {has_secret}')
            except Exception as e:
                print(f'   └─ Could not read: {e}')
        else:
            print(f'✗ Not found: {path}')

    print()
    print('BEST-PRACTICE RECOVERY PATH:')
    print('-' * 70)
    if IN_COLAB:
        print('Path A (recommended): Colab Secrets only')
        print('  1. Keep using browser Colab UI')
        print('  2. Add MASSIVE_ACCESS_KEY and MASSIVE_SECRET_KEY in 🔑 panel')
        print('  3. Enable Notebook access for both')
        print('  4. Re-open this notebook and run Cells 1→5')
        print()
        print('Path B (reliable fallback): Drive .env')
        print('  1. Create .env with MASSIVE_S3_ACCESS_KEY / MASSIVE_S3_SECRET_KEY')
        print('  2. Save to /content/drive/MyDrive/trade-bot/.env')
        print('  3. Re-run credentials cell (Cell 5)')
    else:
        print('Use .env in notebooks folder or environment variables locally.')
else:
    print('✅ Credentials already loaded successfully!')

✅ Credentials already loaded successfully!


In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import json
import pickle
import warnings
import time
from datetime import datetime, timedelta
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import s3fs
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, mean_squared_error,
    classification_report, ConfusionMatrixDisplay,
)
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from lightgbm import LGBMClassifier, LGBMRegressor

warnings.filterwarnings('ignore', category=UserWarning)
print('Imports OK')

## 1 — Load multi-symbol minute-bar data from Massive S3

Load 1-minute bars for multiple liquid symbols, then resample to 5m/15m/1h later.
Pooling 12 symbols over 6+ months targets 100k+ rows per timeframe.

```
flatfiles/us_stocks_sip/minute_aggs_v1/{year}/{month}/{date}.csv.gz
```

In [ ]:
# ── Diagnostic: Test S3 Connection ────────────────────────────────────────────
# Run this cell to verify credentials work BEFORE loading large datasets.
# This is OPTIONAL — skip if credentials are already working.

print('S3 Connection Diagnostic')
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print()

# Check for placeholder credentials
is_placeholder = S3_CONFIG['access_key'].startswith('YOUR_')
if is_placeholder:
    print('❌ PLACEHOLDER CREDENTIALS DETECTED')
    print()
    print('Credentials are still at their default placeholder values.')
    print('You MUST set real credentials before data loading will work.')
    print()
    print('Steps to fix:')
    print('  1. Go to previous credentials cell')
    print('  2. Choose ONE option (Secrets panel, env var, or hardcoded)')
    print('  3. Set real Massive.com credentials')
    print('  4. Re-run credentials cell')
    print('  5. Then re-run THIS diagnostic cell')
else:
    # Try to connect
    print('✅ Real credentials detected. Testing S3 connection...')
    print()
    try:
        fs = s3fs.S3FileSystem(
            key=S3_CONFIG['access_key'],
            secret=S3_CONFIG['secret_key'],
            client_kwargs={'endpoint_url': S3_CONFIG['endpoint']},
        )

        # Try to list a small directory to verify access
        test_path = f"{S3_CONFIG['bucket']}/us_stocks_sip/"
        try:
            files = fs.ls(test_path)[:10]  # Get first 10 entries to verify access
            print(f'✅ SUCCESS: Connected to {S3_CONFIG["endpoint"]}')
            print(f'✅ Can access: {test_path}')
            print(f'✅ Sample files found: {len(files)} entries')
            print()
            print('Credentials are working! You can now run the data loading cell.')
        except PermissionError as e:
            # HTTP 403 Forbidden — credentials problem
            print(f'❌ PERMISSION DENIED (HTTP 403 Forbidden)')
            print(f'   Endpoint: {S3_CONFIG["endpoint"]}')
            print(f'   Path: {test_path}')
            print()
            print('   Possible causes:')
            print('   • Access key is invalid or expired')
            print('   • Secret key is incorrect')
            print('   • Account lacks S3 API permissions')
            print('   • Check Massive.com account settings')
        except Exception as e:
            print(f'❌ ERROR: {type(e).__name__}')
            print(f'   {str(e)[:200]}')
            print()
            print('   Check:')
            print('   • Endpoint URL is correct')
            print('   • Network connection is working')
            print('   • Credentials are correctly formatted')
    except Exception as e:
        print(f'❌ FAILED TO CREATE S3 CLIENT: {type(e).__name__}')
        print(f'   {str(e)[:200]}')

S3 Connection Diagnostic
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

✅ Real credentials detected. Testing S3 connection...

✅ SUCCESS: Connected to https://files.massive.com
✅ Can access: flatfiles/us_stocks_sip/
✅ Sample files found: 4 entries

Credentials are working! You can now run the data loading cell.


In [ ]:
# ── Data loader with parallel S3 downloads — multi-symbol ─────────────────────

KEY_FORMATS = [
    'us_stocks_sip/minute_aggs_v1/{year}/{month:02d}/{date}.csv.gz',
    'us_stocks_sip/minute-aggregates/{year}/{month:02d}/{date}.csv.gz',
]

_fs_cache = {}

def _get_fs(cfg: dict) -> s3fs.S3FileSystem:
    key = cfg['endpoint']
    if key not in _fs_cache:
        _fs_cache[key] = s3fs.S3FileSystem(
            key=cfg['access_key'],
            secret=cfg['secret_key'],
            client_kwargs={'endpoint_url': cfg['endpoint']},
        )
    return _fs_cache[key]


def _load_single_file(args):
    """Load a single date's file from S3 for a given symbol."""
    symbol, date_str, cfg = args
    fs = _get_fs(cfg)
    symbol = symbol.upper()
    for fmt in KEY_FORMATS:
        dt = datetime.strptime(date_str, '%Y-%m-%d')
        path = f"{cfg['bucket']}/{fmt.format(year=dt.year, month=dt.month, date=date_str)}"
        try:
            with fs.open(path, 'rb') as f:
                df = pd.read_csv(f, compression='gzip')
                df.columns = [c.lower() for c in df.columns]
                sym_col = 'ticker' if 'ticker' in df.columns else 'symbol'
                filtered = df[df[sym_col].str.upper() == symbol].copy()
                if not filtered.empty:
                    return 'success', date_str, filtered
        except FileNotFoundError:
            continue
        except PermissionError:
            return 'forbidden', date_str, None
        except Exception:
            return 'error', date_str, None
    return 'notfound', date_str, None


def load_minute_bars(symbol: str, start_date: str, end_date: str, cfg: dict,
                     max_workers: int = 15) -> pd.DataFrame:
    """Load 1-minute bars from S3 for a single symbol."""
    current = datetime.strptime(start_date, '%Y-%m-%d')
    end = datetime.strptime(end_date, '%Y-%m-%d')
    trading_days = []
    while current <= end:
        if current.weekday() < 5:
            trading_days.append(current.strftime('%Y-%m-%d'))
        current += timedelta(days=1)

    print(f'  Loading {symbol} ({len(trading_days)} trading days)...', end=' ')
    dfs = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(_load_single_file, (symbol, d, cfg)): d
            for d in trading_days
        }
        for future in as_completed(futures):
            status, _, df = future.result()
            if status == 'success' and df is not None:
                dfs.append(df)

    if not dfs:
        print('NO DATA')
        return pd.DataFrame()

    result = pd.concat(dfs, ignore_index=True)
    print(f'{len(result):,} bars')
    return result


def load_multi_symbol_minute_bars(symbols, start_date, end_date, cfg):
    """Load 1-minute bars for multiple symbols, tagged with _symbol column."""
    all_dfs = []
    for sym in symbols:
        df = load_minute_bars(sym, start_date, end_date, cfg)
        if not df.empty:
            sym_col = 'ticker' if 'ticker' in df.columns else 'symbol'
            df['_symbol'] = df[sym_col].str.upper()
            all_dfs.append(df)

    if not all_dfs:
        raise ValueError(f'No data found for any symbol in {symbols}')

    result = pd.concat(all_dfs, ignore_index=True)
    print(f'\nTotal: {len(result):,} 1m bars across {len(all_dfs)} symbols')
    return result


print('✅ Multi-symbol data loader defined')

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Pool 12 liquid US equities for sufficient sample size per timeframe
SYMBOLS = ['SPY', 'QQQ', 'IWM', 'DIA', 'AAPL', 'MSFT', 'NVDA', 'AMZN',
           'META', 'GOOGL', 'TSLA', 'AMD']
TIMEFRAMES = ['5m', '15m', '1h']

START_DATE = '2025-06-01'
END_DATE   = '2025-12-31'

# Timeframe config: bars_per_minute, annualization factor, wall-clock conversions
TIMEFRAME_CONFIG = {
    '1m':  {'minutes': 1,  'bars_per_day': 390, 'bars_per_hour': 60},
    '5m':  {'minutes': 5,  'bars_per_day': 78,  'bars_per_hour': 12},
    '15m': {'minutes': 15, 'bars_per_day': 26,  'bars_per_hour': 4},
    '1h':  {'minutes': 60, 'bars_per_day': 7,   'bars_per_hour': 1},
}

# Walk-forward CV wall-clock settings (applied uniformly across timeframes)
CV_PURGE_HOURS = 1      # 1 hour purge gap between train/test
CV_TEST_DAYS   = 5      # 1 trading week test window
CV_N_SPLITS    = 8

# Persistence controls
USE_PERSISTENT_CACHE = True
CACHE_KEY = f"multi_{'_'.join(SYMBOLS[:4])}_{START_DATE}_{END_DATE}".replace('-', '')
RAW_CACHE_PATH = Path(CACHE_DIR) / f'raw_1m_{CACHE_KEY}.pkl'

start_time = time.time()

if USE_PERSISTENT_CACHE and RAW_CACHE_PATH.exists():
    print(f'Loading cached 1m data: {RAW_CACHE_PATH}')
    df_1m = pd.read_pickle(RAW_CACHE_PATH)
else:
    print(f'Loading 1m bars for {len(SYMBOLS)} symbols ({START_DATE} → {END_DATE})...')
    df_1m = load_multi_symbol_minute_bars(SYMBOLS, START_DATE, END_DATE, S3_CONFIG)

    if USE_PERSISTENT_CACHE:
        df_1m.to_pickle(RAW_CACHE_PATH)
        print(f'Cached raw data → {RAW_CACHE_PATH}')

elapsed = time.time() - start_time
print(f'\n⏱️  Load time: {elapsed:.1f}s')
print(f'   Total 1m bars: {len(df_1m):,}')
if '_symbol' in df_1m.columns:
    print(f'   Symbols loaded: {sorted(df_1m["_symbol"].unique())}')
df_1m.head(3)

In [ ]:
# ── Explore the raw data ───────────────────────────────────────────────────────
print(f'Shape:   {df_1m.shape}')
print(f'Columns: {list(df_1m.columns)}')
print()

if '_symbol' in df_1m.columns:
    print('Bars per symbol:')
    for sym, count in df_1m['_symbol'].value_counts().sort_index().items():
        print(f'  {sym}: {count:,}')
    print()

required = ['open', 'high', 'low', 'close', 'volume']
missing = [c for c in required if c not in df_1m.columns]
if missing:
    print(f'WARNING: missing columns {missing}')
else:
    print('All required OHLCV columns present')
    print(df_1m[required].describe().round(4))

In [ ]:
# ── Visualize per-symbol bar counts ───────────────────────────────────────────
if '_symbol' in df_1m.columns:
    counts = df_1m['_symbol'].value_counts().sort_values()
    fig, ax = plt.subplots(figsize=(10, 4))
    counts.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(f'1m Bars per Symbol ({START_DATE} to {END_DATE})')
    ax.set_xlabel('Bar count')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    plt.tight_layout()
    plt.show()

## 2 — Resample & Feature Engineering

Resample 1m bars to 5m, 15m, and 1h candles. Features use **wall-clock-equivalent windows**:

| Wall-clock | 5m bars | 15m bars | 1h bars |
|---|---|---|---|
| ~25 min (short) | 5 | 2 | 1* |
| ~100 min (medium) | 20 | 7 | 2* |
| ~300 min (long) | 60 | 20 | 5 |

*Minimum of 3 bars enforced for rolling calculations.

**Hurst exponent** uses a ~100 min window (scaled by timeframe).

All features are **past-only** — no lookahead.

In [ ]:
# ── Resampling ────────────────────────────────────────────────────────────────

def resample_to_timeframe(df_1m, timeframe):
    """Resample 1-minute bars to a higher timeframe (5m, 15m, 1h)."""
    if timeframe == '1m':
        out = df_1m.copy()
        out['_timeframe'] = '1m'
        return out

    cfg = TIMEFRAME_CONFIG[timeframe]
    minutes = cfg['minutes']

    # Parse timestamp
    if 'window_start' in df_1m.columns:
        ts = pd.to_datetime(df_1m['window_start'], unit='ns')
    elif 'timestamp' in df_1m.columns:
        ts = pd.to_datetime(df_1m['timestamp'])
    else:
        raise ValueError('No timestamp column found')

    df = df_1m.copy()
    df['_ts'] = ts
    df = df.sort_values(['_symbol', '_ts']).reset_index(drop=True)

    resampled_parts = []
    freq = f'{minutes}min'

    for symbol, group in df.groupby('_symbol'):
        g = group.set_index('_ts').sort_index()
        ohlcv = g[['open', 'high', 'low', 'close', 'volume']].resample(freq).agg({
            'open': 'first',
            'high': 'max',
            'low': 'min',
            'close': 'last',
            'volume': 'sum',
        }).dropna(subset=['open'])
        ohlcv['_symbol'] = symbol
        ohlcv['_timeframe'] = timeframe
        ohlcv = ohlcv.reset_index().rename(columns={'_ts': 'datetime'})
        resampled_parts.append(ohlcv)

    result = pd.concat(resampled_parts, ignore_index=True)
    result = result.sort_values(['_symbol', 'datetime']).reset_index(drop=True)
    return result


# ── Hurst helper ──────────────────────────────────────────────────────────────

def _hurst_rs(series):
    """Hurst exponent via R/S analysis — pure numpy."""
    n = len(series)
    if n < 20:
        return 0.5
    mean = series.mean()
    deviation = np.cumsum(series - mean)
    r = deviation.max() - deviation.min()
    s = series.std(ddof=1)
    if s == 0:
        return 0.5
    return float(np.clip(np.log(r / s) / np.log(n), 0.0, 1.0))


# ── Timeframe-aware features ─────────────────────────────────────────────────

def compute_features_timeframe_aware(df, timeframe):
    """Compute features with window sizes scaled to wall-clock time.

    Windows: short=25min, medium=100min, long=300min → bar counts vary by tf.
    """
    cfg = TIMEFRAME_CONFIG[timeframe]
    minutes_per_bar = cfg['minutes']

    windows_minutes = [25, 100, 300]
    windows = [max(3, round(m / minutes_per_bar)) for m in windows_minutes]

    feat = pd.DataFrame(index=df.index)
    log_ret = np.log(df['close'] / df['close'].shift(1))

    for w, wm in zip(windows, windows_minutes):
        label = f'{wm}min'
        feat[f'realized_vol_{label}'] = (
            log_ret.rolling(w, min_periods=w).std() * np.sqrt(252 * cfg['bars_per_day'])
        )
        feat[f'log_return_{label}'] = log_ret.rolling(w, min_periods=1).sum()

    # RSI(14 bars)
    delta = df['close'].diff()
    gain = delta.where(delta > 0, 0.0).rolling(14, min_periods=14).mean()
    loss = (-delta.where(delta < 0, 0.0)).rolling(14, min_periods=14).mean()
    rsi = 100.0 - (100.0 / (1.0 + gain / loss.replace(0, np.nan)))

    feat['rsi_14'] = rsi
    zscore_window = max(10, round(300 / minutes_per_bar))
    rsi_mean = rsi.rolling(zscore_window, min_periods=max(5, zscore_window // 3)).mean()
    rsi_std = rsi.rolling(zscore_window, min_periods=max(5, zscore_window // 3)).std()
    feat['rsi_14_zscore'] = (rsi - rsi_mean) / rsi_std.replace(0, np.nan)
    feat['rsi_14_dev50'] = rsi - 50.0

    # Volume z-score
    vol_window = max(10, round(300 / minutes_per_bar))
    vm = df['volume'].rolling(vol_window, min_periods=max(5, vol_window // 3)).mean()
    vs = df['volume'].rolling(vol_window, min_periods=max(5, vol_window // 3)).std()
    feat['volume_zscore'] = (df['volume'] - vm) / vs.replace(0, np.nan)

    # Price range normalized
    range_window = max(5, round(100 / minutes_per_bar))
    bar_range = (df['high'] - df['low']) / df['close'].replace(0, np.nan)
    feat['price_range_norm'] = bar_range.rolling(range_window, min_periods=max(3, range_window // 4)).mean()

    # Hurst exponent
    hurst_window = max(20, round(100 / minutes_per_bar))
    log_price = np.log(df['close'].replace(0, np.nan).ffill().values)
    hurst_vals = np.full(len(df), np.nan)
    for i in range(hurst_window, len(log_price)):
        seg = log_price[i - hurst_window: i]
        if not np.isnan(seg).any():
            try:
                hurst_vals[i] = _hurst_rs(seg)
            except Exception:
                pass
    feat['hurst'] = hurst_vals

    return feat


# ── Next-return targets ──────────────────────────────────────────────────────

def compute_return_targets(df):
    """Compute next-bar return targets. Label at t+1."""
    log_ret = np.log(df['close'] / df['close'].shift(1))
    next_ret = log_ret.shift(-1)
    targets = pd.DataFrame(index=df.index)
    targets['next_log_return'] = next_ret
    targets['next_direction'] = (next_ret > 0).astype(int)
    return targets


print('✅ Resampling, features, and target functions defined')

In [ ]:
# ── Build datasets for all timeframes ─────────────────────────────────────────

def build_dataset_for_timeframe(df_1m, timeframe, min_rows=100_000):
    """Build features + targets for a single timeframe from 1m bars."""
    print(f'\n{"─"*60}')
    print(f'Building {timeframe} dataset...')

    resampled = resample_to_timeframe(df_1m, timeframe)
    print(f'  Resampled: {len(resampled):,} bars')

    all_X, all_y_dir, all_y_ret, all_last_ret = [], [], [], []
    symbols_used = []

    for symbol, group in resampled.groupby('_symbol'):
        group = group.sort_values('datetime').reset_index(drop=True)
        if len(group) < 200:
            continue

        feat = compute_features_timeframe_aware(group, timeframe)
        targets = compute_return_targets(group)
        log_ret = np.log(group['close'] / group['close'].shift(1))

        combined = pd.concat([feat, targets, log_ret.rename('last_return')], axis=1)
        valid = combined.notna().all(axis=1)
        combined = combined[valid]

        if len(combined) < 50:
            continue

        feature_cols = list(feat.columns)
        all_X.append(combined[feature_cols])
        all_y_dir.append(combined['next_direction'])
        all_y_ret.append(combined['next_log_return'])
        all_last_ret.append(combined['last_return'])
        symbols_used.append(symbol)

    if not all_X:
        raise ValueError(f'No valid data for timeframe {timeframe}')

    X = pd.concat(all_X, ignore_index=True)
    y_dir = pd.concat(all_y_dir, ignore_index=True)
    y_ret = pd.concat(all_y_ret, ignore_index=True)
    last_ret = pd.concat(all_last_ret, ignore_index=True)

    n_rows = len(X)
    status = '✅' if n_rows >= min_rows else '⚠️'
    print(f'  {status} {timeframe}: {n_rows:,} rows from {len(symbols_used)} symbols '
          f'(target: {min_rows:,})')

    return {
        'X': X, 'y_direction': y_dir, 'y_return': y_ret,
        'last_return': last_ret, 'symbols': symbols_used,
        'timeframe': timeframe, 'n_rows': n_rows,
        'feature_columns': list(X.columns),
    }


# Build all three datasets
print('Building datasets for all timeframes...')
datasets = {}
for tf in TIMEFRAMES:
    datasets[tf] = build_dataset_for_timeframe(df_1m, tf)

print(f'\n{"="*60}')
print('DATASET SUMMARY')
print(f'{"Timeframe":<12} {"Rows":<12} {"Symbols":<10} {"Features":<10}')
print('-' * 44)
for tf in TIMEFRAMES:
    d = datasets[tf]
    print(f'{tf:<12} {d["n_rows"]:<12,} {len(d["symbols"]):<10} {len(d["feature_columns"]):<10}')

## 3 — Walk-Forward CV with Baselines

**Timeframe-aware splitter**: gap and test window scale by wall-clock time.

| Setting | Wall-clock | 5m bars | 15m bars | 1h bars |
|---|---|---|---|---|
| Purge gap | 1 hour | 12 | 4 | 1 |
| Test window | 5 days | 390 | 130 | 35 |

**Baselines** evaluated on each test fold:
- **Persistence**: direction = sign(last return), magnitude = last return
- **Random walk**: direction = majority class, magnitude = 0

In [ ]:
# ── Baseline evaluation ───────────────────────────────────────────────────────

def evaluate_baselines(y_true_dir, y_true_ret, last_ret):
    """Evaluate persistence and random-walk baselines."""
    persistence_dir = (last_ret > 0).astype(int)
    persistence_acc = accuracy_score(y_true_dir, persistence_dir)
    persistence_f1 = f1_score(y_true_dir, persistence_dir, average='weighted')
    persistence_mse = mean_squared_error(y_true_ret, last_ret)

    majority = int(y_true_dir.mean() > 0.5)
    rw_dir = np.full_like(y_true_dir, majority)
    rw_acc = accuracy_score(y_true_dir, rw_dir)
    rw_f1 = f1_score(y_true_dir, rw_dir, average='weighted')
    rw_mse = mean_squared_error(y_true_ret, np.zeros_like(y_true_ret))

    return {
        'persistence_accuracy': persistence_acc, 'persistence_f1': persistence_f1,
        'persistence_mse': persistence_mse,
        'random_walk_accuracy': rw_acc, 'random_walk_f1': rw_f1,
        'random_walk_mse': rw_mse,
    }


def make_timeframe_cv(timeframe):
    """Create TimeSeriesSplit with wall-clock-scaled gap and test_size."""
    cfg = TIMEFRAME_CONFIG[timeframe]
    gap_bars = CV_PURGE_HOURS * cfg['bars_per_hour']
    test_bars = CV_TEST_DAYS * cfg['bars_per_day']
    print(f'  CV: gap={gap_bars} bars ({CV_PURGE_HOURS}h), '
          f'test={test_bars} bars ({CV_TEST_DAYS}d), {CV_N_SPLITS} splits')
    return TimeSeriesSplit(n_splits=CV_N_SPLITS, gap=gap_bars, test_size=test_bars)


print('✅ Baseline and CV functions defined')

In [ ]:
# ── Train direction + magnitude models for all timeframes ─────────────────────

def train_direction_model(dataset):
    """Train next-bar direction classifier."""
    X, y, tf = dataset['X'], dataset['y_direction'], dataset['timeframe']

    print(f'\n{"="*60}')
    print(f'DIRECTION model — {tf} ({len(X):,} rows)')
    print(f'{"="*60}')

    tscv = make_timeframe_cv(tf)

    pipe = Pipeline([
        ('scaler', RobustScaler()),
        ('clf', LGBMClassifier(
            n_estimators=500, num_leaves=31, learning_rate=0.1,
            min_child_samples=20, class_weight='balanced', verbose=-1,
        )),
    ])

    print('  Running walk-forward CV...')
    cv_results = cross_validate(
        pipe, X, y, cv=tscv,
        scoring=['accuracy', 'f1_weighted', 'roc_auc'],
    )

    mean_acc = cv_results['test_accuracy'].mean()
    mean_f1 = cv_results['test_f1_weighted'].mean()
    mean_auc = cv_results['test_roc_auc'].mean()
    print(f'  Accuracy: {mean_acc:.4f}  F1: {mean_f1:.4f}  AUC: {mean_auc:.4f}')

    # Baselines on last fold
    last_test_idx = list(tscv.split(X))[-1][1]
    baselines = evaluate_baselines(
        y.iloc[last_test_idx].values,
        dataset['y_return'].iloc[last_test_idx].values,
        dataset['last_return'].iloc[last_test_idx].values,
    )
    lift_acc = mean_acc - baselines['persistence_accuracy']
    lift_f1 = mean_f1 - baselines['persistence_f1']
    print(f'  Persistence: acc={baselines["persistence_accuracy"]:.4f}  '
          f'f1={baselines["persistence_f1"]:.4f}')
    print(f'  Lift vs persistence: acc={lift_acc:+.4f}  f1={lift_f1:+.4f}')

    pipe.fit(X, y)

    return {
        'model': pipe, 'timeframe': tf, 'target': 'direction',
        'cv_accuracy': mean_acc, 'cv_f1': mean_f1, 'cv_roc_auc': mean_auc,
        'cv_accuracy_per_fold': cv_results['test_accuracy'].tolist(),
        'cv_f1_per_fold': cv_results['test_f1_weighted'].tolist(),
        'cv_auc_per_fold': cv_results['test_roc_auc'].tolist(),
        'baselines': baselines, 'lift_accuracy': lift_acc, 'lift_f1': lift_f1,
        'n_rows': dataset['n_rows'], 'symbols': dataset['symbols'],
        'feature_columns': dataset['feature_columns'],
    }


def train_magnitude_model(dataset):
    """Train next-bar return magnitude regressor."""
    X, y, tf = dataset['X'], dataset['y_return'], dataset['timeframe']

    print(f'\n{"="*60}')
    print(f'MAGNITUDE model — {tf} ({len(X):,} rows)')
    print(f'{"="*60}')

    tscv = make_timeframe_cv(tf)

    pipe = Pipeline([
        ('scaler', RobustScaler()),
        ('reg', LGBMRegressor(
            n_estimators=500, num_leaves=31, learning_rate=0.1,
            min_child_samples=20, verbose=-1,
        )),
    ])

    print('  Running walk-forward CV...')
    cv_results = cross_validate(
        pipe, X, y, cv=tscv,
        scoring=['neg_mean_squared_error', 'r2'],
    )

    mean_mse = -cv_results['test_neg_mean_squared_error'].mean()
    mean_r2 = cv_results['test_r2'].mean()
    print(f'  MSE: {mean_mse:.8f}  R2: {mean_r2:.4f}')

    # Baselines on last fold
    last_test_idx = list(tscv.split(X))[-1][1]
    ret_test = y.iloc[last_test_idx].values
    last_ret_test = dataset['last_return'].iloc[last_test_idx].values
    persistence_mse = mean_squared_error(ret_test, last_ret_test)
    rw_mse = mean_squared_error(ret_test, np.zeros_like(ret_test))
    print(f'  Persistence MSE: {persistence_mse:.8f}  RW MSE: {rw_mse:.8f}')

    pipe.fit(X, y)

    return {
        'model': pipe, 'timeframe': tf, 'target': 'magnitude',
        'cv_mse': mean_mse, 'cv_r2': mean_r2,
        'cv_mse_per_fold': (-cv_results['test_neg_mean_squared_error']).tolist(),
        'cv_r2_per_fold': cv_results['test_r2'].tolist(),
        'baseline_persistence_mse': persistence_mse, 'baseline_rw_mse': rw_mse,
        'n_rows': dataset['n_rows'], 'symbols': dataset['symbols'],
        'feature_columns': dataset['feature_columns'],
    }


# Run full experiment matrix
results = {}
for tf in TIMEFRAMES:
    ds = datasets[tf]
    results[tf] = {
        'direction': train_direction_model(ds),
        'magnitude': train_magnitude_model(ds),
    }

## 4 — Experiment Matrix Results

Compare direction (classification) and magnitude (regression) across all three timeframes.
The **lift** column shows how much the model beats the persistence baseline.

In [ ]:
# ── Results comparison table ───────────────────────────────────────────────────
print('='*80)
print('DIRECTION CLASSIFICATION')
print('='*80)
print(f'{"TF":<6} {"Rows":<10} {"Accuracy":<10} {"F1":<10} {"AUC":<10} '
      f'{"Pers.Acc":<10} {"Lift(Acc)":<10} {"Lift(F1)":<10}')
print('-' * 76)
for tf in TIMEFRAMES:
    d = results[tf]['direction']
    print(f'{tf:<6} {d["n_rows"]:<10,} {d["cv_accuracy"]:<10.4f} '
          f'{d["cv_f1"]:<10.4f} {d["cv_roc_auc"]:<10.4f} '
          f'{d["baselines"]["persistence_accuracy"]:<10.4f} '
          f'{d["lift_accuracy"]:<+10.4f} {d["lift_f1"]:<+10.4f}')

print()
print('='*80)
print('MAGNITUDE REGRESSION')
print('='*80)
print(f'{"TF":<6} {"Rows":<10} {"MSE":<14} {"R2":<10} {"Pers.MSE":<14} {"RW MSE":<14}')
print('-' * 68)
for tf in TIMEFRAMES:
    m = results[tf]['magnitude']
    print(f'{tf:<6} {m["n_rows"]:<10,} {m["cv_mse"]:<14.8f} '
          f'{m["cv_r2"]:<10.4f} '
          f'{m["baseline_persistence_mse"]:<14.8f} '
          f'{m["baseline_rw_mse"]:<14.8f}')

# Find best timeframe
best_tf = max(TIMEFRAMES, key=lambda tf: results[tf]['direction']['lift_accuracy'])
best_lift = results[best_tf]['direction']['lift_accuracy']
print(f'\nBest timeframe (by lift over persistence): {best_tf} (lift={best_lift:+.4f})')

beats_baseline = best_lift > 0
if beats_baseline:
    print(f'✅ Model beats persistence baseline on {best_tf}')
else:
    print(f'⚠️  No timeframe beats persistence baseline — consider more data or features')

In [ ]:
# ── Visualization: per-fold CV scores across timeframes ───────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = {'5m': 'steelblue', '15m': 'darkorange', '1h': 'forestgreen'}
folds = range(1, CV_N_SPLITS + 1)

# Accuracy per fold
ax = axes[0]
for tf in TIMEFRAMES:
    d = results[tf]['direction']
    ax.plot(folds, d['cv_accuracy_per_fold'], 'o-', label=tf, color=colors[tf])
    ax.axhline(d['baselines']['persistence_accuracy'], ls=':', color=colors[tf], alpha=0.4)
ax.set_title('Direction: Accuracy per Fold')
ax.set_xlabel('Fold')
ax.set_ylabel('Accuracy')
ax.legend()
ax.set_ylim(0.35, 0.65)

# F1 per fold
ax = axes[1]
for tf in TIMEFRAMES:
    d = results[tf]['direction']
    ax.plot(folds, d['cv_f1_per_fold'], 'o-', label=tf, color=colors[tf])
ax.set_title('Direction: F1 per Fold')
ax.set_xlabel('Fold')
ax.set_ylabel('F1 (weighted)')
ax.legend()

# AUC per fold
ax = axes[2]
for tf in TIMEFRAMES:
    d = results[tf]['direction']
    ax.plot(folds, d['cv_auc_per_fold'], 'o-', label=tf, color=colors[tf])
ax.axhline(0.5, ls='--', color='grey', alpha=0.5, label='chance')
ax.set_title('Direction: ROC-AUC per Fold')
ax.set_xlabel('Fold')
ax.set_ylabel('AUC')
ax.legend()

plt.suptitle('Walk-Forward CV — Direction Classification', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Magnitude plots
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
for tf in TIMEFRAMES:
    m = results[tf]['magnitude']
    ax.plot(folds, m['cv_mse_per_fold'], 'o-', label=tf, color=colors[tf])
ax.set_title('Magnitude: MSE per Fold')
ax.set_xlabel('Fold')
ax.set_ylabel('MSE')
ax.legend()

ax = axes[1]
for tf in TIMEFRAMES:
    m = results[tf]['magnitude']
    ax.plot(folds, m['cv_r2_per_fold'], 'o-', label=tf, color=colors[tf])
ax.axhline(0, ls='--', color='grey', alpha=0.5)
ax.set_title('Magnitude: R2 per Fold')
ax.set_xlabel('Fold')
ax.set_ylabel('R2')
ax.legend()

plt.suptitle('Walk-Forward CV — Magnitude Regression', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5 — Feature Importance

Compare which features matter most at each timeframe.

In [ ]:
# ── Feature importance per timeframe ──────────────────────────────────────────
fig, axes = plt.subplots(1, len(TIMEFRAMES), figsize=(6 * len(TIMEFRAMES), 5))
if len(TIMEFRAMES) == 1:
    axes = [axes]

for ax, tf in zip(axes, TIMEFRAMES):
    model = results[tf]['direction']['model']
    lgbm = model.named_steps['clf']
    feat_imp = pd.Series(
        lgbm.feature_importances_,
        index=results[tf]['direction']['feature_columns'],
    ).sort_values(ascending=True)

    feat_imp.plot(kind='barh', ax=ax, color=colors.get(tf, 'steelblue'))
    ax.set_title(f'{tf} — Direction')
    ax.set_xlabel('Importance')

plt.suptitle('Feature Importance (LightGBM)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 6 — Save Models to Google Drive

In [ ]:
# ── Save all models + metadata to Drive ───────────────────────────────────────
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
saved_files = []

for tf in TIMEFRAMES:
    for target_name in ['direction', 'magnitude']:
        r = results[tf][target_name]
        model_path = Path(SAVE_DIR) / f'return_{target_name}_{tf}_{timestamp}.pkl'
        meta_path = model_path.with_suffix('.json')

        with open(model_path, 'wb') as f:
            pickle.dump(r['model'], f)

        meta = {
            'timeframe': tf,
            'target': target_name,
            'trained_at': timestamp,
            'train_start': START_DATE,
            'train_end': END_DATE,
            'n_rows': r['n_rows'],
            'symbols': r['symbols'],
            'feature_columns': r['feature_columns'],
            'feature_schema_version': '3.0.0',
        }
        if target_name == 'direction':
            meta.update({
                'cv_accuracy': round(r['cv_accuracy'], 4),
                'cv_f1': round(r['cv_f1'], 4),
                'cv_roc_auc': round(r['cv_roc_auc'], 4),
                'baselines': r['baselines'],
                'lift_accuracy': round(r['lift_accuracy'], 4),
                'lift_f1': round(r['lift_f1'], 4),
            })
        else:
            meta.update({
                'cv_mse': float(r['cv_mse']),
                'cv_r2': round(r['cv_r2'], 4),
                'baseline_persistence_mse': float(r['baseline_persistence_mse']),
                'baseline_rw_mse': float(r['baseline_rw_mse']),
            })

        with open(meta_path, 'w') as f:
            json.dump(meta, f, indent=2)

        saved_files.append(model_path.name)
        print(f'  Saved: {model_path.name}')

print(f'\nAll models saved to: {SAVE_DIR}')
print(f'Total files: {len(saved_files) * 2} (pkl + json)')

## 7 — Register on OPC Server

After downloading the .pkl files from Google Drive:

```bash
# On your OPC server
cd /home/opc/projects/trade-bot/market-dashboard/backend

# Register the best direction model
uv run python notebooks/register_model.py \
    --model-path /path/to/return_direction_5m_YYYYMMDD_HHMMSS.pkl \
    --meta-path  /path/to/return_direction_5m_YYYYMMDD_HHMMSS.json

# Restart ml-api to load the new champion
docker-compose -f ../docker-compose.yaml -f ../docker-compose.app.yaml restart ml-api
```

Open MLflow UI at http://localhost:5000 to verify the `@champion` alias was set.

### Notes
- Feature schema version is `3.0.0` (timeframe-aware features)
- Models are per-timeframe — pick the best one based on lift over persistence baseline
- Direction and magnitude models are saved separately
- The metadata JSON records timeframe, symbols, and baseline comparison
